# Part B – Courtesy Amount Recognition
**ICS472 – Natural Language Processing**  
**Team:** Mohammed Al Sheqaih · Abdulrhman Ammar

**Goal:** Train a CRNN (CNN + BiLSTM + CTC) to decode the digit sequence from a courtesy amount crop.

**Evaluation metrics:**
- Digit-level accuracy: `(1 − (I+D+S) / N) × 100`
- % of amounts with no errors
- % of amounts with exactly one error
- % of amounts with two or more errors

> **Prerequisites:** Run `01_Part_A_Detection.ipynb` first — this notebook loads the courtesy crops from `artifacts/crops/`.

## 1. Imports & Setup

In [ ]:
import sys, os, json, random
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from utils import (
    TRAIN_CA, TEST_CA, ARTIFACTS,
    parse_courtesy_amounts, build_vocab, courtesy_summary
)

# ── Device ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams['figure.dpi'] = 120

CROPS_DIR   = ARTIFACTS / 'crops'
MODEL_DIR   = ARTIFACTS / 'courtesy_model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_H       = 32      # fixed crop height after resize
MIN_W       = 64      # minimum crop width (pad if narrower)
HIDDEN      = 256     # BiLSTM hidden size per direction
N_LSTM      = 2       # BiLSTM layers
EPOCHS      = 120
BATCH       = 32
LR          = 1e-3
PATIENCE    = 20      # early stopping

## 2. Vocabulary & Labels

In [ ]:
# Load or rebuild vocabulary
vocab_path = ARTIFACTS / 'ca_vocab.json'
if vocab_path.exists():
    with open(vocab_path, encoding='utf-8') as f:
        vdata = json.load(f)
    ca_vocab = vdata['vocab']
    ca_t2i   = vdata['t2i']
    ca_i2t   = {int(k): v for k, v in vdata['i2t'].items()}
    print(f'Vocabulary loaded from file ({len(ca_vocab)} tokens).')
else:
    # Fallback: build from label files (run 00_EDA.ipynb for the saved version)
    train_ca = parse_courtesy_amounts(TRAIN_CA)
    test_ca  = parse_courtesy_amounts(TEST_CA)
    all_seqs = [
        [t for t in v if t != '<BOS/EOS>']
        for v in list(train_ca.values()) + list(test_ca.values())
    ]
    ca_vocab, ca_t2i, ca_i2t = build_vocab(all_seqs)
    print(f'Vocabulary built inline ({len(ca_vocab)} tokens).')

BLANK  = 0
N_CLASSES = len(ca_vocab)
print(f'Tokens: {ca_vocab}')

In [ ]:
# Parse labels — strip BOS/EOS, keep only inner digit tokens
def load_ca_labels(txt_path):
    raw = parse_courtesy_amounts(txt_path)
    return {
        fname: [t for t in tokens if t != '<BOS/EOS>']
        for fname, tokens in raw.items()
    }

train_labels = load_ca_labels(TRAIN_CA)
test_labels  = load_ca_labels(TEST_CA)

print(f'Train labels: {len(train_labels)}')
print(f'Test  labels: {len(test_labels)}')

# Quick sanity check
for k, v in list(train_labels.items())[:3]:
    print(f'  {k}  →  {v}')

## 3. Dataset & DataLoader

In [ ]:
class CourtesyDataset(Dataset):
    def __init__(self, crops_dir, labels, t2i, img_h=32, min_w=64):
        self.t2i    = t2i
        self.img_h  = img_h
        self.min_w  = min_w
        self.samples = []

        for img_path in sorted(crops_dir.glob('*.tif')):
            key = img_path.name  # e.g. Cac00000.tif
            if key in labels and len(labels[key]) > 0:
                self.samples.append((img_path, labels[key]))

        print(f'Dataset: {len(self.samples)} samples from {crops_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, token_seq = self.samples[idx]
        img = Image.open(img_path).convert('L')

        # Resize to fixed height, maintain aspect ratio
        w, h  = img.size
        new_w = max(self.min_w, int(w * self.img_h / h))
        img   = img.resize((new_w, self.img_h), Image.BILINEAR)

        # To tensor and normalise to [0, 1]; invert so ink = high
        img_t = T.ToTensor()(img)          # (1, H, W), range [0,1]

        # Encode label
        label = torch.tensor(
            [self.t2i[t] for t in token_seq if t in self.t2i],
            dtype=torch.long
        )
        return img_t, label


def collate_fn(batch):
    imgs, labels = zip(*batch)
    # Pad images to max width in batch (pad right with 1.0 = white)
    max_w  = max(img.shape[2] for img in imgs)
    padded = torch.ones(len(imgs), 1, IMG_H, max_w)
    for i, img in enumerate(imgs):
        padded[i, :, :, :img.shape[2]] = img

    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_cat    = torch.cat(labels)  # (sum of lengths,)
    input_lengths = torch.tensor(
        [max_w // 4 for _ in imgs], dtype=torch.long  # T = W // 4 (CNN stride)
    )
    return padded, labels_cat, input_lengths, label_lengths


train_ds = CourtesyDataset(CROPS_DIR / 'train' / 'courtesy', train_labels, ca_t2i, IMG_H, MIN_W)
test_ds  = CourtesyDataset(CROPS_DIR / 'test'  / 'courtesy', test_labels,  ca_t2i, IMG_H, MIN_W)

# 90/10 train/val split
val_size   = int(0.1 * len(train_ds))
train_size = len(train_ds) - val_size
train_sub, val_sub = torch.utils.data.random_split(
    train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_sub, batch_size=BATCH, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_sub,   batch_size=BATCH, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,   batch_size=BATCH, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'Train: {train_size} | Val: {val_size} | Test: {len(test_ds)}')

## 4. CRNN Model

```
Input  (B, 1, 32, W)
  │
  ▼
CNN  →  (B, 512, 1, W//4)    7 conv blocks, batch norm, ReLU
  │
  ▼
Reshape  →  (W//4, B, 512)   height squeezed, width becomes time axis
  │
  ▼
BiLSTM (2 layers, 256 units each direction)  →  (W//4, B, 512)
  │
  ▼
Linear  →  (W//4, B, vocab_size)
  │
  ▼
LogSoftmax  ─►  CTC Loss
```

In [ ]:
def conv_bn_relu(in_ch, out_ch, **kwargs):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, **kwargs),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class CRNN(nn.Module):
    def __init__(self, n_classes, img_h=32, hidden=256, n_lstm=2):
        super().__init__()
        # ── CNN backbone ─────────────────────────────────────────────
        # Input: (B, 1, 32, W)
        self.cnn = nn.Sequential(
            conv_bn_relu(1,   64),
            nn.MaxPool2d(2, 2),                 # → (B, 64, 16, W/2)

            conv_bn_relu(64, 128),
            nn.MaxPool2d(2, 2),                 # → (B, 128, 8, W/4)

            conv_bn_relu(128, 256),             # → (B, 256, 8, W/4)
            conv_bn_relu(256, 256),
            nn.MaxPool2d((2, 1)),               # → (B, 256, 4, W/4)

            conv_bn_relu(256, 512),             # → (B, 512, 4, W/4)
            conv_bn_relu(512, 512),
            nn.MaxPool2d((2, 1)),               # → (B, 512, 2, W/4)

            conv_bn_relu(512, 512),
            nn.MaxPool2d((2, 1)),               # → (B, 512, 1, W/4)
        )
        # ── BiLSTM ───────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=512, hidden_size=hidden,
            num_layers=n_lstm, bidirectional=True,
            batch_first=False, dropout=0.2 if n_lstm > 1 else 0
        )
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        # CNN
        x = self.cnn(x)                        # (B, 512, 1, W//4)
        x = x.squeeze(2)                       # (B, 512, W//4)
        x = x.permute(2, 0, 1)                 # (T, B, 512)
        # BiLSTM
        x, _ = self.lstm(x)                    # (T, B, hidden*2)
        # Projection
        x = self.fc(x)                         # (T, B, n_classes)
        return torch.log_softmax(x, dim=-1)


model = CRNN(N_CLASSES, img_h=IMG_H, hidden=HIDDEN, n_lstm=N_LSTM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CRNN parameters: {total_params:,}')

## 5. Training

In [ ]:
def greedy_decode(log_probs, i2t, blank=0):
    """Greedy CTC decoding: collapse repeats, remove blanks."""
    indices = log_probs.argmax(dim=-1).cpu().tolist()  # (T,)
    tokens, prev = [], None
    for idx in indices:
        if idx != prev:
            tokens.append(idx)
        prev = idx
    return [i2t[i] for i in tokens if i != blank]


def run_epoch(loader, model, criterion, optimiser=None):
    training = optimiser is not None
    model.train() if training else model.eval()
    total_loss = 0
    refs, hyps = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels, in_lens, lbl_lens in loader:
            imgs = imgs.to(DEVICE)

            log_probs = model(imgs)            # (T, B, C)
            T_actual  = log_probs.size(0)
            in_lens   = torch.clamp(in_lens, max=T_actual)

            loss = criterion(
                log_probs, labels.to(DEVICE),
                in_lens.to(DEVICE), lbl_lens.to(DEVICE)
            )

            if training:
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimiser.step()

            total_loss += loss.item()

            # Collect sequences for accuracy computation
            offset = 0
            for i, llen in enumerate(lbl_lens.tolist()):
                ref = [ca_i2t[idx] for idx in labels[offset:offset+llen].tolist()]
                hyp = greedy_decode(log_probs[:, i, :], ca_i2t)
                refs.append(ref)
                hyps.append(hyp)
                offset += llen

    metrics = courtesy_summary(refs, hyps)
    return total_loss / len(loader), metrics['digit_accuracy']


criterion = nn.CTCLoss(blank=BLANK, reduction='mean', zero_infinity=True)
optimiser = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode='max', factor=0.5, patience=7, min_lr=1e-5
)

best_val_acc = -1
patience_ctr = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimiser)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion)
    scheduler.step(vl_acc)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_DIR / 'best.pt')
    else:
        patience_ctr += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:>3} | '
              f'train loss {tr_loss:.4f} acc {tr_acc:.1f}% | '
              f'val loss {vl_loss:.4f} acc {vl_acc:.1f}%')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs).')
        break

print(f'\nBest val digit accuracy: {best_val_acc:.2f}%')

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('CTC Loss')
ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Digit Accuracy (%)')
ax2.set_title('Digit Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(ARTIFACTS / 'partB_training_curves.png', bbox_inches='tight')
plt.show()

## 6. Evaluation on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_DIR / 'best.pt', map_location=DEVICE))
model.eval()

all_refs, all_hyps, all_fnames = [], [], []

with torch.no_grad():
    for imgs, labels, in_lens, lbl_lens in tqdm(test_loader, desc='Test eval'):
        imgs      = imgs.to(DEVICE)
        log_probs = model(imgs)
        T_actual  = log_probs.size(0)
        in_lens   = torch.clamp(in_lens, max=T_actual)

        offset = 0
        for i, llen in enumerate(lbl_lens.tolist()):
            ref = [ca_i2t[idx] for idx in labels[offset:offset+llen].tolist()]
            hyp = greedy_decode(log_probs[:, i, :], ca_i2t)
            all_refs.append(ref)
            all_hyps.append(hyp)
            offset += llen

metrics = courtesy_summary(all_refs, all_hyps)
print('── Test Set Results (Part B) ──')
print(f"  Digit accuracy       : {metrics['digit_accuracy']}%")
print(f"  % no errors          : {metrics['pct_no_error']}%")
print(f"  % one error          : {metrics['pct_one_error']}%")
print(f"  % two or more errors : {metrics['pct_two_plus']}%")

## 7. Error Analysis

In [ ]:
from utils import edit_distance

# Collect per-sample errors
errors = [
    (edit_distance(r, h), r, h)
    for r, h in zip(all_refs, all_hyps)
]
errors_nonzero = [(d, r, h) for d, r, h in errors if d > 0]

print(f'Samples with errors: {len(errors_nonzero)} / {len(errors)}')
print('\nWorst-case examples (highest edit distance):')
for dist, ref, hyp in sorted(errors_nonzero, reverse=True)[:10]:
    print(f'  dist={dist}  ref={"".join(ref)}  hyp={"".join(hyp)}')

In [ ]:
# Error distribution bar chart
dist_counts = [0, 0, 0]   # [0 errors, 1 error, >=2 errors]
for d, _, _ in errors:
    if d == 0:   dist_counts[0] += 1
    elif d == 1: dist_counts[1] += 1
    else:        dist_counts[2] += 1

labels_bar = ['0 errors', '1 error', '≥2 errors']
plt.figure(figsize=(6, 4))
bars = plt.bar(labels_bar, dist_counts, color=['#4caf50', '#ff9800', '#f44336'])
for bar, cnt in zip(bars, dist_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(cnt), ha='center', fontsize=10)
plt.ylabel('Number of samples')
plt.title('Courtesy Recognition — Error Distribution (Test Set)')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'partB_error_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Visual samples: correct and incorrect predictions
test_img_list = sorted((CROPS_DIR / 'test' / 'courtesy').glob('*.tif'))
fname_to_idx  = {p.name: i for i, p in enumerate(test_ds.samples)}

correct   = [(r, h) for d, r, h in errors if d == 0][:3]
incorrect = [(r, h) for d, r, h in errors if d > 0][:3]

fig, axes = plt.subplots(6, 1, figsize=(10, 8))
pairs = [('Correct', c) for c in correct] + [('Incorrect', c) for c in incorrect]
for ax, (label, (ref, hyp)) in zip(axes, pairs):
    ax.text(0.5, 0.5,
            f'[{label}]  ref: {"".join(ref)}   →   pred: {"".join(hyp)}',
            ha='center', va='center', fontsize=11,
            color='green' if label == 'Correct' else 'red',
            transform=ax.transAxes)
    ax.axis('off')
plt.suptitle('Sample Predictions — Courtesy Amount', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'partB_samples.png', bbox_inches='tight')
plt.show()

## 8. Save Predictions

In [ ]:
# Save per-sample predictions for Part D
predictions_out = []
for (img_path, ref_tokens), hyp in zip(test_ds.samples, all_hyps):
    # Derive original image stem: Cac03000.tif → ac03000
    fname = img_path.name                   # Cac03000.tif
    stem  = fname[1:].replace('.tif', '')   # ac03000
    predictions_out.append({
        'file':      fname,
        'stem':      stem,
        'reference': ''.join(ref_tokens),
        'predicted': ''.join(hyp),
    })

with open(ARTIFACTS / 'partB_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(predictions_out, f, indent=2, ensure_ascii=False)

# Save metrics
with open(ARTIFACTS / 'partB_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Predictions and metrics saved to artifacts/')

## 9. Summary

In [ ]:
print('── Part B Complete ──')
print(f"  Model         : CRNN (CNN + BiLSTM×{N_LSTM} + CTC)")
print(f"  Vocab size    : {N_CLASSES} tokens")
print(f"  Parameters    : {total_params:,}")
print(f"  Best val acc  : {best_val_acc:.2f}%")
print()
print(f"  Test results:")
print(f"    Digit accuracy       : {metrics['digit_accuracy']}%")
print(f"    % no errors          : {metrics['pct_no_error']}%")
print(f"    % one error          : {metrics['pct_one_error']}%")
print(f"    % two or more errors : {metrics['pct_two_plus']}%")
print()
print('  → Run 03_Part_C_Legal.ipynb next')